In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
silver_core_df = spark.table(SILVER_CORE_TABLE)

print("Bronze rows:", bronze_df.count())
print("Silver core rows:", silver_core_df.count())

In [0]:
duplicate_event_ids = (
    silver_core_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate event IDs:", duplicate_event_ids)

In [0]:
required_field_check = (
    silver_core_df
    .select(
        F.sum(F.col("event_id").isNull().cast("int"))
            .alias("null_event_id"),

        F.sum(F.col("event_type").isNull().cast("int"))
            .alias("null_event_type"),

        F.sum(F.col("event_ts").isNull().cast("int"))
            .alias("null_event_ts"),

        F.sum(F.col("actor_id").isNull().cast("int"))
            .alias("null_actor_id"),

        F.sum(F.col("repo_id").isNull().cast("int"))
            .alias("null_repo_id")
    )
)

required_field_check.show()

In [0]:
timestamp_check = (
    silver_core_df
    .select(
        F.min("event_ts").alias("min_event_ts"),
        F.max("event_ts").alias("max_event_ts"),

        F.sum(
            (
                F.col("event_ts") <
                F.to_timestamp(F.lit("2026-09-01 00:00:00"))
            ).cast("int")
        ).alias("events_before_expected_window"),

        F.sum(
            (
                F.col("event_ts") >=
                F.to_timestamp(F.lit("2026-09-01 10:00:00"))
            ).cast("int")
        ).alias("events_after_expected_window")
    )
)

timestamp_check.show(truncate=False)

In [0]:
malformed_json_count = (
    bronze_df
    .filter(
        F.get_json_object("raw_json", "$.id").isNull()
    )
    .count()
)

print("Malformed / unparseable Bronze JSON rows:", malformed_json_count)

In [0]:
hour_completeness_df = (
    bronze_df
    .select("source_hour")
    .distinct()
    .orderBy("source_hour")
)

hour_completeness_df.show(30, truncate=False)

print(
    "Distinct source hours:",
    hour_completeness_df.count()
)

In [0]:
hourly_volume_df = (
    bronze_df
    .groupBy("source_hour")
    .agg(
        F.count("*").alias("event_count")
    )
    .orderBy("source_hour")
)

hourly_volume_df.show(
    30,
    truncate=False
)

In [0]:
dq_results = [
    (
        "silver_core_row_count_matches_bronze",
        "FAIL",
        607244 == 607244,
        "Bronze and Silver Core row counts must match"
    ),
    (
        "duplicate_event_ids",
        "FAIL",
        duplicate_event_ids == 0,
        f"duplicate_event_ids={duplicate_event_ids}"
    ),
    (
        "null_event_id",
        "FAIL",
        0 == 0,
        "null_event_id=0"
    ),
    (
        "null_event_type",
        "FAIL",
        0 == 0,
        "null_event_type=0"
    ),
    (
        "null_event_ts",
        "FAIL",
        0 == 0,
        "null_event_ts=0"
    ),
    (
        "null_actor_id",
        "FAIL",
        0 == 0,
        "null_actor_id=0"
    ),
    (
        "null_repo_id",
        "WARN",
        1 == 0,
        "null_repo_id=1"
    ),
    (
        "malformed_bronze_json",
        "FAIL",
        malformed_json_count == 0,
        f"malformed_json_rows={malformed_json_count}"
    ),
    (
        "source_hour_count",
        "FAIL",
        10 == 10,
        "10 expected source hours, 10 observed"
    )
]

dq_summary_df = (
    spark.createDataFrame(
        dq_results,
        ["check_name", "severity", "passed", "details"]
    )
    .withColumn(
        "status",
        F.when(F.col("passed"), F.lit("PASS"))
         .when(F.col("severity") == "WARN", F.lit("WARN"))
         .otherwise(F.lit("FAIL"))
    )
    .select(
        "check_name",
        "severity",
        "status",
        "details"
    )
)

dq_summary_df.show(
    truncate=False
)

In [0]:
bronze_current_df = spark.table(BRONZE_TABLE)
silver_current_df = spark.table(SILVER_CORE_TABLE)

bronze_row_count = bronze_current_df.count()
silver_row_count = silver_current_df.count()

duplicate_event_ids = (
    silver_current_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

silver_null_metrics = (
    silver_current_df
    .agg(
        F.sum(F.col("event_id").isNull().cast("int"))
            .alias("null_event_id"),

        F.sum(F.col("event_type").isNull().cast("int"))
            .alias("null_event_type"),

        F.sum(F.col("event_ts").isNull().cast("int"))
            .alias("null_event_ts"),

        F.sum(F.col("actor_id").isNull().cast("int"))
            .alias("null_actor_id"),

        F.sum(F.col("repo_id").isNull().cast("int"))
            .alias("null_repo_id")
    )
    .first()
)

malformed_json_count = (
    bronze_current_df
    .filter(
        F.get_json_object("raw_json", "$.id").isNull()
    )
    .count()
)

In [0]:
observed_hours_df = (
    bronze_current_df
    .select("source_hour")
    .distinct()
)

hour_bounds = (
    observed_hours_df
    .agg(
        F.min("source_hour").alias("min_hour"),
        F.max("source_hour").alias("max_hour")
    )
    .first()
)

expected_hours_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(hour_bounds["min_hour"]),
                F.lit(hour_bounds["max_hour"]),
                F.expr("INTERVAL 1 HOUR")
            )
        ).alias("source_hour")
    )
)

missing_hours_df = (
    expected_hours_df
    .join(
        observed_hours_df,
        on="source_hour",
        how="left_anti"
    )
)

missing_source_hours = missing_hours_df.count()

print("Missing source hours:", missing_source_hours)

missing_hours_df.orderBy("source_hour").show(
    truncate=False
)

In [0]:
invalid_event_hour_count = (
    silver_current_df
    .filter(
        (F.col("event_ts") < F.col("source_hour"))
        |
        (
            F.col("event_ts") >=
            F.expr("source_hour + INTERVAL 1 HOUR")
        )
    )
    .count()
)

print(
    "Events outside their source hour:",
    invalid_event_hour_count
)

In [0]:
dq_results = [
    (
        "silver_core_row_count_matches_bronze",
        "FAIL",
        silver_row_count == bronze_row_count,
        f"bronze={bronze_row_count}, silver={silver_row_count}"
    ),

    (
        "duplicate_event_ids",
        "FAIL",
        duplicate_event_ids == 0,
        f"duplicate_event_ids={duplicate_event_ids}"
    ),

    (
        "null_event_id",
        "FAIL",
        silver_null_metrics["null_event_id"] == 0,
        f"null_event_id={silver_null_metrics['null_event_id']}"
    ),

    (
        "null_event_type",
        "FAIL",
        silver_null_metrics["null_event_type"] == 0,
        f"null_event_type={silver_null_metrics['null_event_type']}"
    ),

    (
        "null_event_ts",
        "FAIL",
        silver_null_metrics["null_event_ts"] == 0,
        f"null_event_ts={silver_null_metrics['null_event_ts']}"
    ),

    (
        "null_actor_id",
        "FAIL",
        silver_null_metrics["null_actor_id"] == 0,
        f"null_actor_id={silver_null_metrics['null_actor_id']}"
    ),

    (
        "null_repo_id",
        "WARN",
        silver_null_metrics["null_repo_id"] == 0,
        f"null_repo_id={silver_null_metrics['null_repo_id']}"
    ),

    (
        "malformed_bronze_json",
        "FAIL",
        malformed_json_count == 0,
        f"malformed_json_rows={malformed_json_count}"
    ),

    (
        "missing_source_hours",
        "FAIL",
        missing_source_hours == 0,
        f"missing_source_hours={missing_source_hours}"
    ),

    (
        "event_timestamp_matches_source_hour",
        "FAIL",
        invalid_event_hour_count == 0,
        f"events_outside_source_hour={invalid_event_hour_count}"
    )
]

dq_summary_df = (
    spark.createDataFrame(
        dq_results,
        ["check_name", "severity", "passed", "details"]
    )
    .withColumn(
        "status",
        F.when(F.col("passed"), F.lit("PASS"))
         .when(F.col("severity") == "WARN", F.lit("WARN"))
         .otherwise(F.lit("FAIL"))
    )
    .select(
        "check_name",
        "severity",
        "status",
        "details"
    )
)

dq_summary_df.show(
    truncate=False
)

In [0]:
DQ_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.dq_run_results"

dq_run_df = (
    dq_summary_df
    .withColumn(
        "dq_run_ts",
        F.current_timestamp()
    )
    .withColumn(
        "bronze_row_count",
        F.lit(bronze_row_count)
    )
    .withColumn(
        "silver_row_count",
        F.lit(silver_row_count)
    )
)

if not spark.catalog.tableExists(DQ_RESULTS_TABLE):
    (
        dq_run_df
        .write
        .format("delta")
        .saveAsTable(DQ_RESULTS_TABLE)
    )
else:
    (
        dq_run_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(DQ_RESULTS_TABLE)
    )

print("DQ results persisted.")

In [0]:
spark.table(DQ_RESULTS_TABLE) \
    .orderBy(F.desc("dq_run_ts")) \
    .show(20, truncate=False)

In [0]:
dq_fail_count = (
    dq_summary_df
    .filter(F.col("status") == "FAIL")
    .count()
)

dq_warn_count = (
    dq_summary_df
    .filter(F.col("status") == "WARN")
    .count()
)

if dq_fail_count > 0:
    overall_dq_status = "FAIL"
elif dq_warn_count > 0:
    overall_dq_status = "WARN"
else:
    overall_dq_status = "PASS"

print("Overall DQ status:", overall_dq_status)
print("FAIL checks:", dq_fail_count)
print("WARN checks:", dq_warn_count)

## Phase 7 conclusions

Data quality controls implemented:

- Bronze and Silver Core row-count reconciliation
- Event ID uniqueness
- Required Silver Core field checks
- Repository ID null monitoring
- Malformed Bronze JSON detection
- Dynamic missing source-hour detection
- Event timestamp validation against source hour
- Persistent Delta history of DQ results
- Overall PASS / WARN / FAIL evaluation

Current result:

- Overall DQ status: WARN
- FAIL checks: 0
- WARN checks: 1
- The warning is one known ForkEvent with a missing top-level repository ID.
- The record is retained rather than dropped because the event payload remains usable.

Hourly event volume is currently monitored but no anomaly threshold is enforced yet.
A defensible volume baseline will be created after expanding the observation window.